# ViT Experiments - Background Reliance
- Runs the ViT architecture, trained with and without background randomisation agumentation across three seeds

# Importing Libraries

In [3]:
import pathlib
import json
import torch
import numpy as np
import random
import pandas as pd

from cory_data_wrapper import *
from cory_utils import *
import tensorboard
import time

# Select Device

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

Using cuda


# Create constants and create paths

In [5]:
# Make sure that the directory exists
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
SEEDS = [0, 339, 1, 2, 3]
VIT_ARCHITECTURE = ["vit_base_patch16_224.augreg_in1k"]

DATA_DIR = pathlib.Path("data")
DATA_ROUTE = DATA_DIR / "rebuild" / "in9_train"
VAL_ROUTE = DATA_DIR / "rebuild" /"in9_val"
BOUNDING_BOX_PATH = DATA_DIR / "LOC_train_solution.csv"
TEST_INSTANCES = DATA_DIR / "bg_challenge"
RESULTS = pathlib.Path("./results")
TENSORBOARD_DIR = pathlib.Path("./runs")
FIGURES = pathlib.Path("./figures")
RESULTS.mkdir(exist_ok=True)
TENSORBOARD_DIR.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

In [ ]:
print(ViTModel("vit_base_patch16_224.augreg_in1k", num_classes=9, pretrained=False).backbone.pretrained_cfg["mean"])
m = ViTModel("vit_base_patch16_224.augreg_in1k", num_classes=9, pretrained=True)
cfg = m.backbone.pretrained_cfg
print(cfg["architecture"], cfg["tag"])

# Load the data

In [7]:
# Load data
bboxs = load_bbox_csv(BOUNDING_BOX_PATH)
base_train_dataset = IN9BoundingBoxDataset(DATA_ROUTE, bboxs, image_size=IMAGE_SIZE)
train_class_to_idx = base_train_dataset.class_to_idx

# print the instances and num of classes
print(f"Loaded {len(base_train_dataset)} images", "with" , len(base_train_dataset.classes), "classes")

Loaded 69120 images with 9 classes


# Split the data (Train, Validation)

In [8]:
train_subset = base_train_dataset

# Create data loaders for all sets, for faster loading and shuffling and batches

# Create the validation loader
val_dataset_normalised = ImageFolderEval(VAL_ROUTE, image_size=IMAGE_SIZE, mean = VIT_MEAN, std = VIT_STD)
val_loader = make_loader(val_dataset_normalised, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Loaded {len(val_dataset_normalised)} validation images")
assert val_dataset_normalised.class_to_idx == train_class_to_idx
print(val_dataset_normalised.class_to_idx)
# make loaders for the test set, one for each variant
test_variant_loaders = {
    variant: make_loader(
        fix_test_labels(
            ImageFolderEval(TEST_INSTANCES / variant / "val", image_size=IMAGE_SIZE, mean= VIT_MEAN, std= VIT_STD),train_class_to_idx
        ),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )
    for variant in ["original", "no_fg", "mixed_same", "mixed_rand"]
}
print("Test variants loaders created:", list(test_variant_loaders.keys()))

Test variants loaders created: ['original', 'no_fg', 'mixed_same', 'mixed_rand']


# Perform Learning Rate Search

- Perform a small learning rate search on Validation Accuracy
- Train for 3 epochs on 3 different learning rates.

In [6]:
# Create the dictionary for best Learning-Rate
BEST_LR = {
    "vit_base_patch16_224.augreg_in1k": None
}
best_lr_per_arch = {}
# loop through all architectures and perform the Hyper Parameter search for each architecture
for arch_name in VIT_ARCHITECTURE:
    # Check to ensure that it only runs if LR not known,
    if BEST_LR[arch_name] is not None:
        best_lr_per_arch[arch_name] = BEST_LR[arch_name]
        continue

    print(f"\n{'-'*60}\n Hyper Parameter Search: {arch_name} \n{'-'*60}")
    #
    search_train_dataset = BackgroundRandomizationDataset(train_subset, augmentation_strength=0.0, seed=0, mean = VIT_MEAN, std = VIT_STD)
    search_train_loader = make_loader(search_train_dataset,batch_size=BATCH_SIZE, shuffle=True, num_workers=12, persistent_workers=True)

    best_lr, hyperparameter_trials = hyperparameter_search(
        lambda a = arch_name: ViTModel(a, num_classes=9, pretrained=True),
        search_train_loader, val_loader, device,
        learning_rates=(2.5e-5, 5e-5, 1e-4), search_epochs=3,
        log_dir=TENSORBOARD_DIR / "hpsearch" / arch_name
    )
    best_lr_per_arch[arch_name] = best_lr
    print(f"{arch_name} best LR: {best_lr:}")

print("\nBest LR per architecture:", best_lr_per_arch)


Best LR per architecture: {'vit_base_patch16_224': 1e-05}


In [9]:
# Architecture Train, Eval and Test Function

def run_vit_experiment(arch_name ="vit_base_patch16_224.augreg_in1k", augmented = False, seed= 0):
    """
    Run the ViT experiment for a specified architecture and seed.
    Stores the data in results

    @param arch_name: Name of the CNN architecture.
    @param augmented: Whether to use augmented or original data.
    @param seed: Seed for reproducibility.
    @return: A dictionary containing the results of the specified experiment.
    """
    data_type = "augmented" if augmented else "original"
    run_name = f"{arch_name}_{data_type}_{seed}"
    print(f"\n{'='*60}\n{run_name}\n{'='*60}")

    # If data is augmented, strength = 1 (swap) else 0 (do not swap background), swap the background with a random image's background
    train_dataset_for_run = BackgroundRandomizationDataset(
        train_subset, augmentation_strength=1.0 if augmented else 0.0, seed=seed, mean = VIT_MEAN, std = VIT_STD
    )
    train_loader = make_loader(train_dataset_for_run, batch_size=BATCH_SIZE, shuffle=True, num_workers=12, persistent_workers=True)

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    # Define the model and move it to the device
    model = ViTModel(arch_name, num_classes=9, pretrained=True).to(device)

    start_time = time.time()
    # Train the model and store the results
    trained_model, history = train_model(
        model, train_loader, val_loader, device, epochs=EPOCHS, early_stopping_patience=3, model_name=run_name, lr=best_lr_per_arch[arch_name], log_dir=TENSORBOARD_DIR / run_name
    )
    training_time_in_secs = time.time() - start_time

    # Save the model weights
    weights_path = RESULTS / f"{run_name}_weights.pt"
    torch.save(trained_model.state_dict(), weights_path)

    variant_accuracies = evaluate_on_variants(trained_model, test_variant_loaders, device)
    # compute the reliance metrics
    reliance = compute_reliance_metrics(variant_accuracies, num_classes=9)
    print(f"Test variants: {variant_accuracies}")
    print(f"BG-Gap: {reliance['bg_gap']:.4f}, No-FG: {reliance['no_fg_accuracy']:.4f}, Original: {reliance['original_accuracy']:.4f}")
    # Store the results in a dictionary
    results = {
        "architecture" : arch_name,
        "data_type" : data_type,
        "seed" : seed,
        "variant_accuracies" : variant_accuracies,
        "reliance_metrics" : reliance,
        "final_val_accuracy" : history["val_accuracy"][-1],
        "epochs_run" : len(history["val_accuracy"]),
        "training_time_in_secs" : training_time_in_secs,
        "best_val_accuracy" : max(history["val_accuracy"]),
    }
    # Save the dadta
    results_path = RESULTS / "vit_results.json"
    results_each_run = {}
    if results_path.exists():
        with open(results_path, "r") as f:
            for r in json.load(f):
                key = f"{r['architecture']}_{r['data_type']}_{r['seed']}"
                results_each_run[key] = r
    results_each_run[run_name] = results
    with open(results_path, "w") as f:
        json.dump(list(results_each_run.values()), f, indent=2)

    return results

# ViT Base (No Augmentation)

In [ ]:
run_vit_experiment(arch_name= "vit_base_patch16_224.augreg_in1k",augmented=False, seed=0)

In [ ]:
run_vit_experiment(arch_name= "vit_base_patch16_224.augreg_in1k",augmented=False, seed=339)

In [9]:
run_vit_experiment(arch_name= "vit_base_patch16_224.augreg_in1k",augmented=False, seed=1)


vit_base_patch16_224_original_339


C:\Users\minio\PycharmProjects\JupyterProject2\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:1194: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[vit_base_patch16_224_original_339] Epoch 1/10: train_loss=0.4648  train_acc=0.8419 val_loss=0.0247  val_acc=0.9922
[vit_base_patch16_224_original_339] Epoch 2/10: train_loss=0.0243  train_acc=0.9947 val_loss=0.0282  val_acc=0.9918
[vit_base_patch16_224_original_339] Epoch 3/10: train_loss=0.0106  train_acc=0.9977 val_loss=0.0371  val_acc=0.9941
[vit_base_patch16_224_original_339] Epoch 4/10: train_loss=0.0070  train_acc=0.9984 val_loss=0.0494  val_acc=0.9922
[vit_base_patch16_224_original_339] Epoch 5/10: train_loss=0.0048  train_acc=0.9989 val_loss=0.0562  val_acc=0.9913
[vit_base_patch16_224_original_339] Epoch 6/10: train_loss=0.0019  train_acc=0.9996 val_loss=0.0464  val_acc=0.9936
[vit_base_patch16_224_original_339] Early stopping at epoch 6
Test variants: {'original': 0.994567901234568, 'no_fg': 0.7817283950617284, 'mixed_same': 0.974320987654321, 'mixed_rand': 0.9365432098765432}
BG-Gap: 0.0378, No-FG: 0.7817, Original: 0.9946


{'architecture': 'vit_base_patch16_224',
 'data_type': 'original',
 'seed': 339,
 'variant_accuracies': {'original': 0.994567901234568,
  'no_fg': 0.7817283950617284,
  'mixed_same': 0.974320987654321,
  'mixed_rand': 0.9365432098765432},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.7817283950617284,
  'bg_gap': 0.0377777777777778,
  'original_accuracy': 0.994567901234568},
 'final_val_accuracy': 0.9936342592592593,
 'epochs_run': 6,
 'training_time_in_secs': 1759.617758989334,
 'best_val_accuracy': 0.9940682870370371}

In [ ]:
run_vit_experiment(arch_name= "vit_base_patch16_224.augreg_in1k",augmented=False, seed=2)


In [ ]:
run_vit_experiment(arch_name= "vit_base_patch16_224.augreg_in1k",augmented=False, seed=3)


# ViT Base (Augmentation)

In [ ]:
run_vit_experiment(arch_name= "vit_base_patch16_224.augreg_in1k",augmented=True, seed=0)

In [ ]:
run_vit_experiment(arch_name= "vit_base_patch16_224.augreg_in1k",augmented=True, seed=339)

In [12]:
run_vit_experiment(arch_name= "vit_base_patch16_224.augreg_in1k",augmented=True, seed=1)


vit_base_patch16_224_augmented_339
[vit_base_patch16_224_augmented_339] Epoch 1/10: train_loss=0.7377  train_acc=0.7537 val_loss=0.0581  val_acc=0.9870
[vit_base_patch16_224_augmented_339] Epoch 2/10: train_loss=0.1462  train_acc=0.9613 val_loss=0.0527  val_acc=0.9857
[vit_base_patch16_224_augmented_339] Epoch 3/10: train_loss=0.0912  train_acc=0.9761 val_loss=0.0962  val_acc=0.9755
[vit_base_patch16_224_augmented_339] Epoch 4/10: train_loss=0.0624  train_acc=0.9846 val_loss=0.0593  val_acc=0.9864
[vit_base_patch16_224_augmented_339] Early stopping at epoch 4
Test variants: {'original': 0.9908641975308642, 'no_fg': 0.6424691358024691, 'mixed_same': 0.9753086419753086, 'mixed_rand': 0.9676543209876544}
BG-Gap: 0.0077, No-FG: 0.6425, Original: 0.9909


{'architecture': 'vit_base_patch16_224',
 'data_type': 'augmented',
 'seed': 339,
 'variant_accuracies': {'original': 0.9908641975308642,
  'no_fg': 0.6424691358024691,
  'mixed_same': 0.9753086419753086,
  'mixed_rand': 0.9676543209876544},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.6424691358024691,
  'bg_gap': 0.007654320987654284,
  'original_accuracy': 0.9908641975308642},
 'final_val_accuracy': 0.9864004629629629,
 'epochs_run': 4,
 'training_time_in_secs': 2234.904985666275,
 'best_val_accuracy': 0.9869791666666666}

In [ ]:
run_vit_experiment(arch_name= "vit_base_patch16_224.augreg_in1k",augmented=True, seed=2)

In [ ]:
run_vit_experiment(arch_name= "vit_base_patch16_224.augreg_in1k",augmented=True, seed=3)


# Short Summary

In [14]:
results_path = RESULTS / "vit_results.json"
with open(results_path, "r") as f:
    results = json.load(f)

rows = []
for r in results:
    row = {"architecture": r["architecture"], "data_type": r["data_type"], "seed": r["seed"], "best_val_accuracy": r["best_val_accuracy"], "epochs_run": r["epochs_run"]}
    row.update(r["reliance_metrics"])
    row["original_accuracy"] = r["variant_accuracies"].get("original")
    rows.append(row)

df = pd.DataFrame(rows)
summary = df.groupby(["architecture", "data_type"]).agg(
    mean_bg_gap =("bg_gap", "mean"), std_bg_gap =("bg_gap", "std"),
    mean_no_fg =("no_fg_accuracy", "mean"), std_no_fg =("no_fg_accuracy", "std"),
    mean_original =("original_accuracy", "mean"), std_original =("original_accuracy", "std"),
)
summary

mean_bg_gap  std_bg_gap  mean_no_fg  \
architecture         data_type                                        
vit_base_patch16_224 augmented     0.006173    0.001307    0.661811   
                     original      0.044198    0.005582    0.777119   

                                std_no_fg  mean_original  std_original  
architecture         data_type                                          
vit_base_patch16_224 augmented   0.052029       0.990535      0.001267  
                     original    0.005616       0.995226      0.000754

# Show TensorBoard

In [15]:
# %load_ext tensorboard
# %tensorboard --logdir ./runs


# Attention Rollout

In [10]:
from cory_attention_rollout_test import get_vit_attention_rollout
from PIL import Image
import pathlib
import torch

model = ViTModel("vit_base_patch16_224.augreg_in1k", num_classes=9, pretrained=False)
# Print out the parameter count
print(f"ViT-B/16 Parameter Count: {sum(p.numel() for p in model.parameters()):,}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS = pathlib.Path("./results")

mixed_rand_test = fix_test_labels(ImageFolderEval(TEST_INSTANCES / "mixed_rand" / "val", image_size=IMAGE_SIZE, mean = VIT_MEAN, std=VIT_STD), train_class_to_idx)

def perform_attn_rollout(augmented=False, img_index=339):
    if augmented:
        model.load_state_dict(torch.load(RESULTS / "vit_base_patch16_224.augreg_in1k_augmented_0_weights.pt"))
    else:
        model.load_state_dict(torch.load(RESULTS / "vit_base_patch16_224.augreg_in1k_original_0_weights.pt"))
    model.to(device)
    image, label = mixed_rand_test[img_index]

    overlay, predicted = get_vit_attention_rollout(model, image, device = device)
    print(f"True Label: {label}, Predicted: {predicted}")
    Image.fromarray(overlay).save(FIGURES / f"attn_rollout_{'augmented' if augmented else 'original'}_{img_index}.png")

perform_attn_rollout(augmented=False, img_index=2282)
perform_attn_rollout(augmented= True, img_index=2282)

ViT-B/16 Parameter Count: 85,805,577
True Label: 4, Predicted: 0
True Label: 4, Predicted: 4


# Inference Time Counter

In [ ]:
import torch.utils.benchmark as benchmark
# https://docs.pytorch.org/docs/2.14/benchmark_utils.html
# https://docs.pytorch.org/tutorials/recipes/recipes/benchmark.html

def benchmark_model(model,name):
    model=model.cuda().eval()
    x = torch.randn(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device = "cuda")
    torch.cuda.reset_peak_memory_stats()
    timer = benchmark.Timer(stmt="with torch.no_grad(): model(x)", globals={"model": model, "x": x, "torch": torch})
    result = timer.blocked_autorange()
    print(f"{name}: took {result.median*1000} ms, with a peak memory usage of: {torch.cuda.max_memory_allocated() / 1e6} MegaBytes")

benchmark_model(ViTModel("vit_base_patch16_224.augreg_in1k", num_classes=9, pretrained=True), "vit_base_patch16_224.augreg_in1k")

In [17]:
# import torch, random
# from cory_data_wrapper import grabcut_mask_from_bbox, build_mask_from_bbox
#
# def diagnose_augmentation(dataset, n_samples=500, seed=0, min_donor_background_fraction=0.3, use_grabcut=True):
#     py_rng = random.Random(seed)
#     n = len(dataset)
#     indices = py_rng.sample(range(len(dataset)), n)
#
#     n_swapped = 0
#     n_grabcut_success = 0
#
#     for idx in indices:
#         image, label, bbox = dataset[idx]
#         H, W = image.shape[1], image.shape[2]
#
#         if use_grabcut:
#             _, grabcut_succeeded = grabcut_mask_from_bbox(image, bbox)
#             if grabcut_succeeded:
#                 n_grabcut_success += 1
#
#         swapped_this_one = False
#         for _ in range(5):
#             bg_index = py_rng.randrange(len(dataset))
#             if bg_index == idx and len(dataset) > 1:
#                 continue
#             _, _, candidate_bbox = dataset[bg_index]
#             candidate_mask = build_mask_from_bbox(candidate_bbox, (H, W))
#             if candidate_mask.mean().item() >= min_donor_background_fraction:
#                 swapped_this_one = True
#                 break
#         if swapped_this_one:
#             n_swapped += 1
#
#     print(f"Sampled {len(dataset)} images:")
#     print(f"  Background swap actually applied: {n_swapped}/{n} ({n_swapped/n:.1%})")
#     print(f"  unchanged (all 5 donors rejected): {n - n_swapped}/{n} ({(n-n_swapped)/n:.1%})")
#     if use_grabcut:
#         print(f"  GrabCut succeeded: {n_grabcut_success}/{n} ({n_grabcut_success/n:.1%})")
#         print(f"  GrabCut fell back to bbox: {n - n_grabcut_success}/{n} ({(n-n_grabcut_success)/n:.1%})")
#
# diagnose_augmentation(train_subset, n_samples=500, seed=0)